In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import json
import pandas as pd
from collections import Counter
from scipy.stats import gaussian_kde
from copy import deepcopy
from constants import data_dir, meta_data_dir, blues, reds, output_dir, font_prop

In [ ]:
with open(meta_data_dir / Path("task_types.json"), "r") as f:
    task_types = json.load(f)

colors = {
    "pos": reds[3],
    "neg": blues[2],
}
edge_colors = {
    "pos": reds[-2],
    "neg": blues[-2],
}
legends = {
    "pos": "positive",
    "neg": "negtive",
}

# AA distribution

In [ ]:
os.makedirs(save_dir := (output_dir / Path("cls_aa")), exist_ok=True)

In [ ]:
for dataset_name, task_type in task_types.items():
    if task_type != "classification":
        continue
    data_info = pd.read_csv(
        data_dir / Path(dataset_name) / Path("enhanced_data.csv")
    )
    seqs = {"pos": [], "neg": []}
    [
        (seqs["pos"] if row["label"] else seqs["neg"]).append(
            row["sequence"].strip()
        )
        for _, row in data_info.iterrows()
    ]
    aa_counts = {
        label: Counter("".join(seqs_per_label))
        for label, seqs_per_label in seqs.items()
    }
    sorted_aa_types = list(
        sorted(
            set.union(
                *(
                    set(aa_counts_per_label.keys())
                    for _, aa_counts_per_label in aa_counts.items()
                )
            )
        )
    )
    aa_type_to_index = {
        aa_type: index for index, aa_type in enumerate(sorted_aa_types)
    }
    sorted_counts = {
        label: [aa_counts_per_label.get(aa_type, 0) for aa_type in sorted_aa_types]
        for label, aa_counts_per_label in aa_counts.items()
    }
    x = np.arange(len(sorted_aa_types))
    plt.figure(figsize=(16, 6))
    for label, counts_per_label in sorted_counts.items():
        plt.bar(
            x,
            counts_per_label,
            color=colors[label],
            width=1.0,
            align="center",
            edgecolor=edge_colors[label],
            alpha=0.5,
            label=legends[label],
        )

    plt.xticks(x, sorted_aa_types)
    plt.xlabel("Amino Acid Type", fontsize=14, fontproperties=font_prop)
    plt.ylabel("Occurrence", fontsize=14, fontproperties=font_prop)
    plt.xticks(fontsize=13, fontproperties=font_prop)
    plt.yticks(fontsize=13, fontproperties=font_prop)

    legend_font_prop = deepcopy(font_prop)
    legend_font_prop.set_size(14)
    plt.legend(fontsize=20, prop=legend_font_prop)
    # plt.grid(True, linestyle='--', color='gray', alpha=0.5)
    plt.savefig(
        save_dir / Path(f"{dataset_name}.png"),
        dpi=200,
        bbox_inches="tight",
        transparent=True,
    )

# Properties

In [ ]:
os.makedirs(save_dir := (output_dir / Path("cls_prop")), exist_ok=True)

def get_prop(prop_name, row):
    if prop_name == "seq_len":
        return len(row["sequence"].strip())
    elif prop_name == "hydro":
        return row["hydrophobicity"]
    elif prop_name == "charge":
        return row["charge"]
    else:
        raise ValueError

x_axis_names = {
    'seq_len': "Sequence Length",
    'hydro': "Hydrophilicity",
    'charge': "Charge",
}

In [ ]:
prop_names = ['seq_len', 'hydro', 'charge']

for dataset_name, task_type in task_types.items():
    props = {prop_name: {"pos": [], "neg": []} for prop_name in prop_names}
    if task_type != "classification":
        continue
    data_info = pd.read_csv(
        data_dir / Path(dataset_name) / Path("enhanced_data.csv")
    )

    for _, row in data_info.iterrows():
        for prop_name in prop_names:
            (props[prop_name]["pos"] if row["label"] else props[prop_name]["neg"]).append(
                get_prop(prop_name, row)
            )
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for i, prop_name in enumerate(prop_names):
        for label in ['pos', 'neg']:
            prop_values = np.asarray(props[prop_name][label])
            p_prop = gaussian_kde(prop_values)

            counts, bins, _ = axes[i].hist(
                prop_values, 
                bins=50, 
                alpha=0.5, 
                color=colors[label],
                density=False,
                label=legends[label],
            )
            legend_font_prop = deepcopy(font_prop)
            legend_font_prop.set_size(14)
            axes[i].legend(fontsize=20, prop=legend_font_prop)
            
            bin_width = bins[1] - bins[0]
            y_min, y_max = prop_values.min(), prop_values.max()
            x = np.linspace(y_min, y_max, 500)
            y = p_prop(x) * len(prop_values) * bin_width

            axes[i].plot(
                x,
                y,
                color=edge_colors[label],
                linewidth=2,
                alpha=0.8,
            )

        axes[i].set_xlabel(x_axis_names[prop_name], fontsize=14, fontproperties=font_prop)
        if i == 0:
            axes[i].set_ylabel("Occurrence", fontsize=14, fontproperties=font_prop)
        axes[i].tick_params(axis='x', labelsize=13)
        axes[i].tick_params(axis='y', labelsize=13)
        axes[i].grid(True, linestyle='--', color='gray', alpha=0.5)
    plt.savefig(
        save_dir / Path(f"{dataset_name}.png"),
        dpi=200,
        bbox_inches="tight",
        transparent=True,
    )